In [ ]:
# --- Human-in-the-loop: pause the graph mid-run with interrupt() (NOTES.md §4, §5) ---
from typing import Annotated
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt
from langchain.chat_models import init_chat_model
from IPython.display import Image, display

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)
# A checkpointer is MANDATORY for human-in-the-loop: interrupt freezes the graph and
# saves state so it can resume later — that only works because state was persisted
# (NOTES.md §5).
memory_saver=MemorySaver()

@tool 
def human_assistance(query:str)->str:
    """Request assistance from a human."""
    # interrupt() = PAUSING (a different mechanism from a conditional edge, which only
    # ROUTES — NOTES.md §4). It freezes the whole graph, hands control back to your
    # code, and whatever you later pass via Command(resume=...) becomes its return value.
    human_response=interrupt({"query": query})
    return human_response

tool = TavilySearch(max_result=2)
tools=[tool, human_assistance]
llm=init_chat_model("ollama:qwen3:8b")
llm_with_tools=llm.bind_tools(tools)

def chatbot(state:State):
    message=llm_with_tools.invoke(state["messages"])
    return {"messages":[message]}

graph_builder.add_node("chatbot",chatbot)
tool_node=ToolNode(tools)
graph_builder.add_node("tools",tool_node)
graph_builder.add_conditional_edges(   # routing: chatbot -> tools (if tool_calls) else END
    "chatbot",
    tools_condition
)
graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge(START,"chatbot")
graph=graph_builder.compile(checkpointer=memory_saver)
config={"configurable":{"thread_id":"1"}}
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
user_input_msg="I need some expert guidance for building an AI agent. Could you request assistance for me?"
config={"configurable":{"thread_id":"1"}}

events=graph.stream(
    {"messages":user_input_msg},
    config,
    stream_mode="values"
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

I need some expert guidance for building an AI agent. Could you request assistance for me?
================================== Ai Message ==================================
Tool Calls:
  human_assistance (807b99a4-ae1a-47be-9f2d-9ec2913bd733)
 Call ID: 807b99a4-ae1a-47be-9f2d-9ec2913bd733
  Args:
    query: I need some expert guidance for building an AI agent. Could you request assistance for me?
================================== Ai Message ==================================
Tool Calls:
  human_assistance (807b99a4-ae1a-47be-9f2d-9ec2913bd733)
 Call ID: 807b99a4-ae1a-47be-9f2d-9ec2913bd733
  Args:
    query: I need some expert guidance for building an AI agent. Could you request assistance for me?


In [ ]:
# Resume the paused graph. The human's answer (collected seconds or hours later) is
# fed back via Command(resume=...), which becomes the return value of interrupt().
# The thread_id + checkpointer is what lets the conversation survive that gap (NOTES.md §4).
human_response_msg=(
    "We, the experts are here to help! We'd recommend you check out LangGraph to build your agent"
    "It's much more reliable and extensible than simple autonomous agents"
)

human_command=Command(resume={"data": human_response_msg})
events=graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()